# Jour 2 · Obtenir des sorties structurées


## Quand une phrase ne suffit plus

Une réponse destinée à être lue peut rester du texte. Une réponse destinée à un programme doit avoir une forme prévisible : une catégorie, une priorité, un résumé et éventuellement une liste de sources.

Une **sortie structurée** transforme la réponse en objet Python dont les champs sont connus à l'avance.


![Texte libre comparé à un objet structuré](../assets/jour_01/07_sortie_libre_vs_schema.png)

*Pydantic valide la forme des données que le reste du programme devra utiliser.*


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


## Pydantic décrit le contrat

Une classe Pydantic ressemble à une fiche à remplir. Les annotations de type indiquent les valeurs permises ; `Field` explique le sens du champ au modèle et au développeur.


In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class TicketAnalyse(BaseModel):
    categorie: Literal["vpn", "ecran", "imprimante", "securite", "autre"] = Field(
        description="Sujet principal du ticket."
    )
    priorite: Literal["basse", "normale", "haute", "critique"] = Field(
        description="Urgence déduite uniquement des faits fournis."
    )
    resume: str = Field(description="Résumé factuel en une phrase.")


print(TicketAnalyse.model_json_schema())


## Relier le schéma au modèle

`with_structured_output` demande au modèle de respecter le schéma puis transforme le résultat en instance Pydantic. La méthode `json_schema` utilise le format structuré proposé par Ollama.


In [ ]:
analyseur = llm.with_structured_output(TicketAnalyse, method="json_schema")

ticket = (
    "Depuis 08:15, environ 400 personnes ne peuvent plus se connecter au VPN. "
    "Le portail affiche une erreur avant la demande MFA."
)
analyse = analyseur.invoke(ticket)

print(type(analyse))
print(analyse.model_dump())
print("Catégorie :", analyse.categorie)
print("Priorité  :", analyse.priorite)


## Une forme valide n'est pas forcément un fait vrai

Pydantic peut confirmer que `priorite` appartient à une liste autorisée. Il ne peut pas prouver que la priorité choisie correspond à la politique de l'entreprise.

Les règles critiques doivent rester dans du code ou une source métier contrôlée.


### À vous de jouer — définir un second contrat

Complétez le schéma d'une réponse RAG avec un statut explicite et des descriptions non ambiguës.


In [ ]:
# À vous de jouer — ajoutez `statut` et décrivez précisément chaque champ.
class ReponseSupport(BaseModel):
    reponse: str = Field(description="Texte de réponse")
    sources: list[str] = Field(default_factory=list)


print(ReponseSupport.model_json_schema())


## Pourquoi séparer en deux passes ?

Demander simultanément de comprendre un texte long, raisonner, classer et fabriquer une structure augmente le nombre de contraintes dans un seul appel.

Nous séparons deux résultats observables : une extraction factuelle, puis une classification structurée.


![Message brut transformé en faits puis en objet Pydantic](../assets/jour_01/07_chaine_deux_passes.png)

*Chaque passe possède une entrée, une sortie et un objectif vérifiables.*


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


prompt_faits = ChatPromptTemplate.from_messages([
    (
        "system",
        "Extrait uniquement les faits observables. N'ajoute aucune cause ni solution. "
        "Réponds en trois puces maximum.",
    ),
    ("human", "{ticket}"),
])
extraire_faits = prompt_faits | llm | StrOutputParser()

faits = extraire_faits.invoke({"ticket": ticket})
print("PASSE 1 — FAITS")
print(faits)

analyse_deux_passes = analyseur.invoke(faits)
print("\nPASSE 2 — OBJET")
print(analyse_deux_passes.model_dump())


## Ne pas confondre décomposition et pensée privée

Nous n'avons pas besoin d'exiger un long « raisonnement étape par étape ». Il suffit de définir des sorties utiles : faits extraits, décision, justification courte ou objet validé.

Cette approche rend le système plus facile à tester sans dépendre d'un monologue interne invérifiable.


## Déboguer une sortie structurée

Avec `include_raw=True`, LangChain peut retourner le message brut, l'objet parsé et une éventuelle erreur de parsing. C'est utile pendant le développement, mais le contenu brut ne doit pas être traité comme une donnée validée.


In [ ]:
analyseur_debug = llm.with_structured_output(
    TicketAnalyse,
    method="json_schema",
    include_raw=True,
)
debug = analyseur_debug.invoke("Une personne signale un écran noir depuis ce matin.")

print("Objet parsé :", debug["parsed"])
print("Erreur       :", debug["parsing_error"])


## À retenir

- Le schéma est un contrat de forme.
- Les types et descriptions doivent être explicites.
- La validation ne prouve pas la vérité métier.
- Deux passes sont utiles lorsque deux tâches ont des contrats différents.
- Un résultat intermédiaire doit rester concis et vérifiable.
